In [1]:
%pip install -q sentence-transformers chromadb groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

In [2]:
import torch
import chromadb
import sentence_transformers
import groq

print("PyTorch:", torch.__version__)
print("ChromaDB:", chromadb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Groq:", groq.__version__)

print("\nAll required libraries imported successfully!")

PyTorch: 2.11.0+cpu
ChromaDB: 1.5.9
Sentence Transformers: 5.6.0
Groq: 1.6.0

All required libraries imported successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving knowledge_base.zip to knowledge_base.zip


In [4]:
import zipfile
from pathlib import Path

RAG_DIR = Path("/content/rag")
KB_DIR = RAG_DIR / "knowledge_base"
VECTOR_DIR = RAG_DIR / "vector_store"

KB_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile("/content/knowledge_base.zip", "r") as zip_ref:
    zip_ref.extractall(KB_DIR)

print("Knowledge base extracted.")

Knowledge base extracted.


In [5]:
for file in KB_DIR.rglob("*"):
    if file.is_file():
        print(file)

/content/rag/knowledge_base/knowledge_base/investigation_guidelines.md
/content/rag/knowledge_base/knowledge_base/cms_program_integrity.md
/content/rag/knowledge_base/knowledge_base/billing_concepts.md
/content/rag/knowledge_base/knowledge_base/fraud_indicator.md


In [6]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("Groq API key loaded:", bool(GROQ_API_KEY))

Groq API key loaded: True


In [7]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized.")

Groq client initialized.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [9]:
texts = [
    "High provider reimbursement may require investigation.",
    "Unusual claim utilization can indicate potential fraud."
]

embeddings = embedding_model.encode(texts)

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 384)


In [10]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="healthcare_fraud_knowledge"
)

print("ChromaDB collection created.")

ChromaDB collection created.


In [12]:
documents = []
metadatas = []
ids = []

for file in KB_DIR.rglob("*.md"):
    text = file.read_text(encoding="utf-8")

    documents.append(text)

    metadatas.append({
        "source": file.name
    })

    ids.append(file.stem)

print("Documents loaded:", len(documents))

for file in KB_DIR.rglob("*.md"):
    print("Loaded:", file)

Documents loaded: 4
Loaded: /content/rag/knowledge_base/knowledge_base/investigation_guidelines.md
Loaded: /content/rag/knowledge_base/knowledge_base/cms_program_integrity.md
Loaded: /content/rag/knowledge_base/knowledge_base/billing_concepts.md
Loaded: /content/rag/knowledge_base/knowledge_base/fraud_indicator.md


In [13]:
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

print("Embeddings created.")
print("Shape:", embeddings.shape)

Embeddings created.
Shape: (4, 384)


In [14]:
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Knowledge base stored in ChromaDB.")

Knowledge base stored in ChromaDB.


In [15]:
query = """
Provider has a high fraud probability, high anomaly score,
unusually high reimbursement and high claim frequency.
What investigation indicators should be reviewed?
"""

In [16]:
query_embedding = embedding_model.encode(
    [query]
)[0]

In [17]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

In [18]:
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(doc[:1500])


--- Retrieved Document 1 ---
# Healthcare Fraud and Suspicious Pattern Indicators

## Purpose

This document describes patterns that may warrant additional review in healthcare claims and provider data.

These indicators are not proof of fraud. A single indicator should not be interpreted independently. Investigators should consider the provider's specialty, patient population, clinical context, and comparison with appropriate peers.

---

## 1. High Reimbursement

### Description

A provider may warrant additional review when reimbursement amounts are substantially higher than expected for comparable providers or services.

### Possible signals

- High total reimbursement
- High average reimbursement per claim
- High reimbursement per beneficiary
- Repeated high-value claims

### Investigation questions

- Are the high-value claims supported by the underlying services?
- Are reimbursement levels consistent with comparable providers?
- Are particular procedures responsible for the inc

In [19]:
case_input = {
    "provider_id": "PRV10025",

    "model_a": {
        "fraud_probability": 0.88
    },

    "model_b": {
        "anomaly_score": 0.94
    },

    "risk_engine": {
        "final_risk_score": 0.91,
        "risk_level": "HIGH"
    },

    "shap_explanation": [
        {
            "feature": "IP_Total_Reimbursement",
            "value": 1850000,
            "shap_value": 0.21,
            "direction": "increases_risk"
        },
        {
            "feature": "OP_Claim_Count",
            "value": 142,
            "shap_value": 0.16,
            "direction": "increases_risk"
        },
        {
            "feature": "Claims_Per_Beneficiary",
            "value": 3.8,
            "shap_value": 0.11,
            "direction": "increases_risk"
        }
    ]
}

In [20]:
import json

retrieved_context = "\n\n".join(
    results["documents"][0]
)

prompt = f"""
You are an AI assistant supporting a healthcare
fraud investigation.

Analyze the following case using the model evidence,
SHAP explanation and retrieved domain knowledge.

Do not confirm fraud.
Do not invent facts.
Treat model scores as risk signals.
The final decision belongs to a human investigator.

CASE:

Provider ID:
{case_input["provider_id"]}

Model A Fraud Probability:
{case_input["model_a"]["fraud_probability"]:.2%}

Model B Anomaly Score:
{case_input["model_b"]["anomaly_score"]:.2%}

Final Risk Score:
{case_input["risk_engine"]["final_risk_score"]:.2%}

Risk Level:
{case_input["risk_engine"]["risk_level"]}

SHAP Explanation:
{json.dumps(case_input["shap_explanation"], indent=2)}

RETRIEVED KNOWLEDGE:
{retrieved_context}

Generate a structured investigator report with:

1. Executive Risk Summary
2. Model Evidence
3. Key Contributing Factors
4. Explanation of Detected Patterns
5. Relevant Domain Context
6. Recommended Investigation Checks
7. Evidence Sources
8. Investigation Recommendation
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare fraud investigation
support assistant. Be factual and evidence-based.
Never claim that fraud is confirmed.
"""
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    max_tokens=2000
)

report = response.choices[0].message.content

print(report)

**Investigator Report**

### 1. Executive Risk Summary
The provider PRV10025 has been identified with a HIGH risk level, based on a final risk score of 91.00%. This score is derived from Model A's fraud probability of 88.00% and Model B's anomaly score of 94.00%. The high risk score suggests that this provider warrants additional investigation to determine if there are any irregularities in their billing practices.

### 2. Model Evidence
- **Model A Fraud Probability:** 88.00%
- **Model B Anomaly Score:** 94.00%
- **Final Risk Score:** 91.00%
- **Risk Level:** HIGH

### 3. Key Contributing Factors
According to the SHAP explanation, the key factors contributing to the high risk score are:
- **IP_Total_Reimbursement:** $1,850,000, with a SHAP value of 0.21, indicating an increase in risk.
- **OP_Claim_Count:** 142 claims, with a SHAP value of 0.16, indicating an increase in risk.
- **Claims_Per_Beneficiary:** 3.8 claims per beneficiary, with a SHAP value of 0.11, indicating an increase i